In [8]:
import joblib
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix


df = pd.read_csv("credit_card_default_under_1mb.csv")


X = df.drop(columns=['ID', 'default payment next month'])
y = df['default payment next month']
feature_names = X.columns.tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
num_cols = [c for c in feature_names if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', 'passthrough', cat_cols)
    ]
)

print(f"Dataset successfully loaded: {df.shape}")
print(f"Training features count: {len(feature_names)}")

Dataset successfully loaded: (9000, 25)
Training features count: 23


In [9]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

param_dist = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [6, 8, 12, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best Hyperparameters:", search.best_params_)

Best Hyperparameters: {'classifier__n_estimators': 200, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 12}


In [10]:
y_proba = best_model.predict_proba(X_test)[:, 1]

cost_fn = 500  # Cost of missed default
cost_fp = 50   # Cost of rejecting a good applicant

thresholds = np.linspace(0.1, 0.9, 81)
best_threshold = 0.5
min_cost = float('inf')

for th in thresholds:
    y_pred = (y_proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_cost = (fn * cost_fn) + (fp * cost_fp)
    if total_cost < min_cost:
        min_cost = total_cost
        best_threshold = round(float(th), 2)

print(f"Optimal Decision Threshold: {best_threshold}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

Optimal Decision Threshold: 0.16
ROC-AUC Score: 0.7687


In [11]:
model_package = {
    "model": best_model,
    "threshold": best_threshold,
    "feature_names": feature_names
}
joblib.dump(model_package, "model_package.joblib")

metadata = {
    "optimal_threshold": best_threshold,
    "cost_fn": cost_fn,
    "cost_fp": cost_fp,
    "feature_names": feature_names
}
with open("threshold.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved model_package.joblib and threshold.json successfully!")

Saved model_package.joblib and threshold.json successfully!


In [12]:
joblib.dump({"model": best_model, "threshold": best_threshold, "feature_names": feature_names}, "model_package.joblib")

['model_package.joblib']